# Urdu chunking study: complete experiment notebook

Resumable Colab notebook for the study *"How should Urdu documents be chunked for retrieval?"*
No LLM generation or judging anywhere: BM25, open embedding encoders and deterministic metrics only.

**How to use**
1. Runtime > Change runtime type > GPU (A100/T4). Run the config cell, then "Run all".
2. First run with `DRY_RUN = True` (about 5 min). It proves the pipeline on 2 articles and prints all PASS/FAIL lines.
3. Set `DRY_RUN = False` and run everything. Stages can be subset with `STAGES` / `MODELS`.

**Approximate runtime per stage (A100-class GPU, estimates, not measured)**
| Stage | What | Estimate |
|---|---|---|
| A | download, corpus build, self-tests | 5-10 min |
| B | BM25 grid (150 runs) + normalization (12 runs) | 30-120 min (char BM25 dominates; auto-reduces if projected over 2 h) |
| C | dense Urdu: bge ~90 min, qwen ~60 min, e5 ~45 min, labse ~10 min | 3-4 h total, run one model per session if needed |
| D | English control (bge) | 30 min |
| E | hybrid RRF (no embedding) | 10-15 min |
| F | native set (only if files present) | 20-40 min |

**After a disconnect:** reconnect, rerun the config cell and then "Run all". Every finished run is already
in `runs.jsonl` / `run_index.jsonl` / `state.json` on Drive and is skipped. At most the run in progress is lost.

**Outputs** are written to `/content/drive/MyDrive/urdu_chunking/` (or `/content/urdu_chunking/`). The last cell zips them.
Deviations from the spec, if any, are listed in the **DEVIATIONS** section at the very end.

In [ ]:
# ============================== 0. CONFIG (edit me) ==============================
STAGES = ["A", "B", "C", "D", "E", "F"]        # stage A always runs (data + self-tests)
MODELS = ["bge", "e5", "qwen", "labse"]         # dense models for stage C. D uses bge, F uses bge+qwen (if in MODELS)
DRY_RUN = True                                  # 2 articles, stages A/B + one dense config + RRF, under ~5 min
RUN_SPLITS = ["test"]                           # configs are frozen: never tune on test. Add "dev" only for extra checks
USE_DRIVE = True                                # mount Google Drive for resumability
CHAR_GRID_MODE = "auto"                         # "full" | "reduced" (overlap 0 only) | "auto" (reduce if projected > budget)
CHAR_TIME_BUDGET_SEC = 2 * 3600
EN_BOUNDARIES_FOR_EN_CORPUS = False             # False = strict spec (Urdu terminators on English text too). True = also use English periods
STRICT_COUNTS = True                            # abort if corpus counts differ from the spec by > 0.5%
SEED = 0

In [ ]:
# ============================== 1. INSTALL, IMPORTS, PATHS, HELPERS ==============================
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *a])
_pip("-U", "sentence-transformers>=3.0", "transformers>=4.51", "rank_bm25", "pyarrow")

import os, re, json, time, math, gc, random, string, hashlib, zipfile, itertools, unicodedata, urllib.request, shutil
from pathlib import Path
from collections import Counter, defaultdict, namedtuple
import numpy as np
import pandas as pd
import scipy.sparse as sp
from scipy.stats import spearmanr
import torch

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
T0 = time.time()
def log(*a): print(f"[{(time.time()-T0)/60:7.1f}m]", *a, flush=True)

DEVIATIONS = []
def deviation(msg):
    if msg not in DEVIATIONS:
        DEVIATIONS.append(msg); log("DEVIATION:", msg)

def _setup_workdir():
    base = None
    if USE_DRIVE:
        try:
            from google.colab import drive
            drive.mount("/content/drive")
            base = Path("/content/drive/MyDrive/urdu_chunking")
        except Exception as e:
            print("Drive not mounted:", repr(e))
    if base is None:
        base = Path("/content/urdu_chunking")
    base.mkdir(parents=True, exist_ok=True)
    return base

WORK = _setup_workdir()
if DRY_RUN:
    WORK = WORK / "dry_run"
for sub in ("parts", "cache"):
    (WORK / sub).mkdir(parents=True, exist_ok=True)
RAW = Path("/content/urdu_raw"); RAW.mkdir(parents=True, exist_ok=True)
RUNS_JSONL = WORK / "runs.jsonl"
INDEX_JSONL = WORK / "run_index.jsonl"
STATE_JSON = WORK / "state.json"
log("work dir:", WORK, "| DRY_RUN =", DRY_RUN)

def _np_default(o):
    if isinstance(o, (np.integer,)): return int(o)
    if isinstance(o, (np.floating,)): return float(o)
    if isinstance(o, (np.bool_,)): return bool(o)
    if isinstance(o, np.ndarray): return o.tolist()
    raise TypeError(type(o))

def dumps(o): return json.dumps(o, ensure_ascii=False, separators=(",", ":"), default=_np_default)

def append_line(path, line):
    with open(path, "a", encoding="utf-8") as f:
        f.write(line + "\n"); f.flush()
        try: os.fsync(f.fileno())
        except Exception: pass

DONE, INDEX = set(), {}
def load_state():
    DONE.clear(); INDEX.clear()
    if INDEX_JSONL.exists():
        for line in open(INDEX_JSONL, encoding="utf-8"):
            line = line.strip()
            if not line: continue
            try: r = json.loads(line)
            except Exception: continue
            INDEX[r["run_id"]] = r; DONE.add(r["run_id"])
    log(f"resume: {len(DONE)} completed runs found")
load_state()

def save_state():
    tmp = STATE_JSON.with_suffix(".tmp")
    tmp.write_text(json.dumps({"completed": sorted(DONE), "updated": time.strftime("%Y-%m-%dT%H:%M:%S")}))
    tmp.replace(STATE_JSON)

def sha256_file(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for b in iter(lambda: f.read(1 << 20), b""): h.update(b)
    return h.hexdigest()

TESTS = {}
AUX_JSON = WORK / "aux.json"
def report_test(name, ok, detail=""):
    TESTS[name] = {"pass": bool(ok), "detail": detail}
    print(("PASS" if ok else "FAIL"), "|", name, "|", detail, flush=True)

In [ ]:
# ============================== 2. MODELS, TEXT UTILITIES, NORMALIZATION ==============================
TASK = "Given a web search query, retrieve relevant passages that answer the query"
INSTR = "Instruct: " + TASK + "\nQuery: {q}"
MODEL_SPECS = {
    "bge":   dict(hf="BAAI/bge-m3", max_seq=8192, prefix=None, tok_kwargs={}),
    "e5":    dict(hf="intfloat/multilingual-e5-large-instruct", max_seq=512, prefix=INSTR, tok_kwargs={}),
    "qwen":  dict(hf="Qwen/Qwen3-Embedding-0.6B", max_seq=32768, prefix=INSTR, tok_kwargs={"padding_side": "left"}),
    "labse": dict(hf="sentence-transformers/LaBSE", max_seq=256, prefix=None, tok_kwargs={}),
}
MODEL_REV, FERTILITY = {}, {}
TOKS = {}
def get_tokenizer(mk):
    if mk not in TOKS:
        from transformers import AutoTokenizer
        t = AutoTokenizer.from_pretrained(MODEL_SPECS[mk]["hf"], use_fast=True)
        t.model_max_length = 10**9
        TOKS[mk] = t
    return TOKS[mk]

def model_revision(hf_id):
    try:
        from huggingface_hub import constants
        p = Path(constants.HF_HUB_CACHE) / ("models--" + hf_id.replace("/", "--")) / "refs" / "main"
        return p.read_text().strip() if p.exists() else None
    except Exception:
        return None

def fmt_query(mk, q, use_prefix=True):
    p = MODEL_SPECS[mk]["prefix"]
    return p.replace("{q}", q) if (use_prefix and p) else q

WORD_RE = re.compile(r"\S+")
PUNCT_STRIP = string.punctuation + "۔؟،؛"
def clean_tok(w): return w.strip(PUNCT_STRIP).lower()
def term_list(cw, variant):
    if not cw: return []
    if variant == "ws" or len(cw) < 3: return [cw]
    return [cw[i:i + n] for n in (3, 4, 5) for i in range(len(cw) - n + 1)]

# ---- normalization with index maps ----
DIAC = set(chr(c) for c in range(0x064B, 0x0656)) | {chr(0x0670)}
TATWEEL = "\u0640"
N2_REPL = {"ي": "ی", "ك": "ک", "ة": "ۃ", "،": ",", "؛": ";", "?": "؟"}
for _i in range(10):
    N2_REPL[chr(0x0660 + _i)] = str(_i); N2_REPL[chr(0x06F0 + _i)] = str(_i)
ZW = {"\u200c", "\u200d"}

def normalize_with_map(text, variant):
    """Returns (normalized_text, idxmap) where idxmap[i] = original index of normalized char i. N0: idxmap None."""
    if variant == "N0": return text, None
    out, idx = [], []
    prev_space = False
    n2 = variant == "N2"
    for i, ch in enumerate(text):
        if ch in DIAC or ch == TATWEEL: continue
        if n2 and ch in ZW: continue
        if ch != "\n" and ch.isspace():
            if prev_space: continue
            out.append(" "); idx.append(i); prev_space = True; continue
        prev_space = False
        out.append(N2_REPL.get(ch, ch) if n2 else ch); idx.append(i)
    return "".join(out), np.asarray(idx, dtype=np.int64)

def normalize_text(t, variant): return normalize_with_map(t, variant)[0]
def map_span(idxmap, s, e):
    return int(np.searchsorted(idxmap, s, "left")), int(np.searchsorted(idxmap, e, "left"))

In [ ]:
# ============================== 3. CHUNKERS (all operate on word index ranges) ==============================
Cfg = namedtuple("Cfg", "strategy size ov budget")
UR_END = set("۔؟!")
CLOSERS = set("\"'”’»)]}›")
LEAD_STRIP = "\"'“‘(["

def _core_last(w):
    k = len(w)
    while k > 0 and w[k - 1] in CLOSERS: k -= 1
    return w[k - 1] if k > 0 else ""

class DocFeat:
    def __init__(self, doc, en_bound=False):
        ms = list(WORD_RE.finditer(doc))
        n = len(ms); self.n = n
        self.words = [m.group() for m in ms]
        self.ws = np.fromiter((m.start() for m in ms), np.int64, n)
        self.we = np.fromiter((m.end() for m in ms), np.int64, n)
        pb = np.zeros(n, bool)
        if n:
            for m in re.finditer("\n\n", doc):
                k = int(np.searchsorted(self.we, m.start(), "right")) - 1
                if k >= 0: pb[k] = True
            pb[-1] = True
        end_ur = np.zeros(n, bool); end_en = np.zeros(n, bool)
        W = self.words
        for k, w in enumerate(W):
            c = _core_last(w)
            if c in UR_END: end_ur[k] = True
            if c and c in "?!": end_en[k] = True
            elif c == ".":
                if pb[k] or k == n - 1: end_en[k] = True
                else:
                    nx = W[k + 1].lstrip(LEAD_STRIP)
                    if nx[:1].isupper(): end_en[k] = True
        self.pb = pb
        self.end_en = end_en
        self.end_ur = (end_ur | end_en) if en_bound else end_ur

def units_from_flags(flags, n):
    ends = np.flatnonzero(flags)
    out, s = [], 0
    for e in ends:
        out.append((s, int(e) + 1)); s = int(e) + 1
    if s < n: out.append((s, n))
    return out

def split_long(units, limit, piece):
    out = []
    for a, b in units:
        if b - a > limit:
            x = a
            while x < b:
                y = min(x + piece, b); out.append((x, y)); x = y
        else: out.append((a, b))
    return out

def pack_units(units, target, ov_words):
    chunks, i, n = [], 0, len(units)
    while i < n:
        j, w = i, 0
        while j < n:
            uw = units[j][1] - units[j][0]
            if j > i and w + uw > target: break
            w += uw; j += 1
        chunks.append((units[i][0], units[j - 1][1]))
        if j >= n: break
        k, ow = j, 0
        while k - 1 > i:
            uw = units[k - 1][1] - units[k - 1][0]
            if ow + uw <= ov_words: k -= 1; ow += uw
            else: break
        i = k
    return chunks

def chunk_fixed_words(f, size, ov):
    n = f.n; step = max(1, round(size * (1 - ov))); out, i = [], 0
    while i < n:
        j = min(i + size, n); out.append((i, j))
        if j >= n: break
        i += step
    return out

def chunk_fixed_tokens(f, budget, ov, tok_cum):
    n = f.n; out, i = [], 0
    while i < n:
        j = int(np.searchsorted(tok_cum, tok_cum[i] + budget, side="right")) - 1
        j = min(max(j, i + 1), n); out.append((i, j))
        if j >= n: break
        i += max(1, round((j - i) * (1 - ov)))
    return out

def chunk_sentence_aware(f, target, ov):
    u = split_long(units_from_flags(f.end_ur | f.pb, f.n), 2 * target, target)
    return pack_units(u, target, int(round(target * ov)))

def chunk_recursive_english(f, size, ov):
    u = split_long(units_from_flags(f.end_en | f.pb, f.n), size, size)
    return pack_units(u, size, int(round(size * ov)))

def chunk_recursive_urdu(f, size, ov):
    u = split_long(units_from_flags(f.end_ur | f.pb, f.n), size, size)
    return pack_units(u, size, int(round(size * ov)))

In [ ]:
# ============================== 4. NORMALIZED CORPUS CLASS, CHUNK SETS ==============================
class ChunkSet:
    def __init__(self, doc, cs, ce, cw, gwi, gwj):
        self.doc, self.cs, self.ce, self.cw, self.gwi, self.gwj = doc, cs, ce, cw, gwi, gwj
    def __len__(self): return len(self.doc)
    def spans(self): return np.stack([self.doc, self.cs, self.ce], 1).astype(np.int32)

class NCorpus:
    def __init__(self, base, norm):
        self.tag, self.lang, self.norm = base["tag"], base["lang"], norm
        self.docs, self.dsplit, self.dtitle = [], [], []
        self.q_id, self.q_text, self.q_ans, self.q_split = [], [], [], []
        qdoc, qs, qe, qvalid = [], [], [], []
        self.n_total_q = 0; self.fail_q = 0
        for di, d in enumerate(base["docs"]):
            if norm == "N0": nd, mp = d["doc"], None
            else: nd, mp = normalize_with_map(d["doc"], norm)
            self.docs.append(nd); self.dsplit.append(d["split"]); self.dtitle.append(d["title"])
            for q in d["qs"]:
                self.n_total_q += 1
                if mp is None:
                    ns, ne = q["s"], q["e"]; ans = q["answer"]
                else:
                    ns, ne = map_span(mp, q["s"], q["e"]); ans = normalize_text(q["answer"], norm)
                ok = (0 <= ns < ne <= len(nd)) and nd[ns:ne] == ans
                if not ok: self.fail_q += 1
                self.q_id.append(q["qid"]); self.q_text.append(normalize_text(q["question"], norm)); self.q_ans.append(ans)
                self.q_split.append(d["split"])
                qdoc.append(di); qs.append(ns); qe.append(ne); qvalid.append(ok)
        self.qdoc = np.array(qdoc, np.int64); self.qs = np.array(qs, np.int64); self.qe = np.array(qe, np.int64)
        self.qvalid = np.array(qvalid, bool); self.q_split = np.array(self.q_split)
        self._feat, self._tc, self._sp, self._wofs = {}, {}, {}, None
        self.en_bound = (self.lang == "en" and EN_BOUNDARIES_FOR_EN_CORPUS)
        if self.fail_q:
            log(f"[{self.tag}/{norm}] span-mapping failures: {self.fail_q}/{self.n_total_q} ({self.fail_q/max(1,self.n_total_q):.4%}); excluded")

    def split_docs(self, split):
        return np.array([i for i, s in enumerate(self.dsplit) if split == "all" or s == split], np.int64)
    def split_q(self, split):
        return np.flatnonzero(np.ones(len(self.q_split), bool) if split == "all" else (self.q_split == split))
    def feat(self, di):
        if di not in self._feat: self._feat[di] = DocFeat(self.docs[di], self.en_bound)
        return self._feat[di]
    @property
    def wofs(self):
        if self._wofs is None:
            n = [self.feat(i).n for i in range(len(self.docs))]
            self._wofs = np.concatenate([[0], np.cumsum(n)]).astype(np.int64)
        return self._wofs
    def tok_cum(self, di, tkey):
        d = self._tc.setdefault(tkey, {})
        if di not in d:
            f = self.feat(di); tok = get_tokenizer(tkey)
            enc = tok(self.docs[di], add_special_tokens=False, return_offsets_mapping=True, truncation=False)
            off = np.asarray(enc["offset_mapping"], np.int64).reshape(-1, 2)
            starts = off[off[:, 1] > off[:, 0]][:, 0]
            d[di] = np.concatenate([np.searchsorted(starts, f.ws, "left"), [len(starts)]]).astype(np.int64)
        return d[di]

    # ---- sparse (BM25) index over word types ----
    def sparse_index(self, variant):
        if variant in self._sp: return self._sp[variant]
        _ = self.wofs
        t2id, type_terms, vocab = {}, [], {}
        pos_type = np.empty(int(self.wofs[-1]), np.int32); p = 0
        for di in range(len(self.docs)):
            for w in self.feat(di).words:
                tid = t2id.get(w)
                if tid is None:
                    tid = len(t2id); t2id[w] = tid
                    type_terms.append([vocab.setdefault(t, len(vocab)) for t in term_list(clean_tok(w), variant)])
                pos_type[p] = tid; p += 1
        lens = np.array([len(x) for x in type_terms], np.int64)
        rows = np.repeat(np.arange(len(type_terms)), lens)
        cols = np.fromiter(itertools.chain.from_iterable(type_terms), np.int64, count=int(lens.sum()))
        T = sp.csr_matrix((np.ones(len(rows), np.float32), (rows, cols)), shape=(len(type_terms), len(vocab)))
        self._sp[variant] = (pos_type, T, vocab)
        return self._sp[variant]

CORP = {}
BASE = {}
def get_corpus(tag, norm):
    if (tag, norm) not in CORP: CORP[(tag, norm)] = NCorpus(BASE[tag], norm)
    return CORP[(tag, norm)]

def make_chunks(corp, doc_ids, cfg, tkey=None):
    D, CS, CE, CW, GI, GJ = [], [], [], [], [], []
    for di in doc_ids:
        f = corp.feat(int(di))
        if f.n == 0: continue
        s = cfg.strategy
        if s == "fixed_words": r = chunk_fixed_words(f, cfg.size, cfg.ov)
        elif s == "fixed_tokens": r = chunk_fixed_tokens(f, cfg.budget, cfg.ov, corp.tok_cum(int(di), tkey))
        elif s == "sentence_aware": r = chunk_sentence_aware(f, cfg.size, cfg.ov)
        elif s == "recursive_english": r = chunk_recursive_english(f, cfg.size, cfg.ov)
        elif s == "recursive_urdu": r = chunk_recursive_urdu(f, cfg.size, cfg.ov)
        else: raise ValueError(s)
        a = np.asarray(r, np.int64).reshape(-1, 2)
        D.append(np.full(len(a), di, np.int64)); CS.append(f.ws[a[:, 0]]); CE.append(f.we[a[:, 1] - 1])
        CW.append(a[:, 1] - a[:, 0]); GI.append(a[:, 0] + corp.wofs[di]); GJ.append(a[:, 1] + corp.wofs[di])
    cat = lambda x: np.concatenate(x) if x else np.zeros(0, np.int64)
    return ChunkSet(cat(D), cat(CS), cat(CE), cat(CW), cat(GI), cat(GJ))

# ---- grids ----
OVS = [0.0, 0.10, 0.25]
STRATS = ["fixed_words", "fixed_tokens", "sentence_aware", "recursive_english", "recursive_urdu"]
BM25_SIZES = [50, 100, 200, 400, 800]
BM25_TOKENIZER_KEY = "bge"
def cfg_for(strategy, size, ov): return Cfg(strategy, None, ov, size) if strategy == "fixed_tokens" else Cfg(strategy, size, ov, None)
def bm25_grid(): return [cfg_for(s, z, o) for s in STRATS for z in BM25_SIZES for o in OVS]
def dense_core_grid():
    g = [Cfg("fixed_words", z, o, None) for z in (50, 100, 200) for o in OVS]
    g += [Cfg("fixed_tokens", None, 0.0, b) for b in (64, 128, 256)]
    for s in ("sentence_aware", "recursive_english", "recursive_urdu"): g += [Cfg(s, z, 0.0, None) for z in (50, 100, 200)]
    return g
def dense_ext_grid(): return [Cfg(s, z, o, None) for s in ("fixed_words", "sentence_aware", "recursive_urdu") for z in (400, 800) for o in (0.0, 0.25)]
def labse_grid(): return [Cfg(s, z, 0.0, None) for s in ("fixed_words", "sentence_aware", "recursive_urdu") for z in (50, 100)]
def norm_grid(): return [Cfg("fixed_words", z, 0.0, None) for z in (50, 100, 200)]

CORPUS_NAME = {"ur": "uqa_ur", "en": "squad_en", "nat": "native_ur"}
def make_run_id(tag, split, mk, cfg, norm, retr):
    size = cfg.budget if cfg.strategy == "fixed_tokens" else cfg.size
    return f"{tag}_{split}_{mk}_{cfg.strategy}_{size}_{int(round(cfg.ov*100))}_{norm}_{retr}"

In [ ]:
# ============================== 5. METRICS ==============================
def compute_nrel(ch, qdoc, qs, qe):
    nrel = np.zeros(len(qdoc), np.int32)
    if len(qdoc) == 0: return nrel
    order = np.argsort(qdoc, kind="stable"); sd = qdoc[order]
    groups = np.split(order, np.flatnonzero(np.diff(sd)) + 1)
    for g in groups:
        d = qdoc[g[0]]
        lo = np.searchsorted(ch.doc, d, "left"); hi = np.searchsorted(ch.doc, d, "right")
        if hi == lo: continue
        cs = ch.cs[lo:hi][None, :]; ce = ch.ce[lo:hi][None, :]
        nrel[g] = ((cs <= qs[g][:, None]) & (ce >= qe[g][:, None])).sum(1)
    return nrel

BUDGETS = (200, 500, 1000, 2000)
def score_rankings(ch, qdoc, qs, qe, R):
    """R: int array [nq,K] of chunk ids (-1 padded), best first. Relevance is article-scoped."""
    nrel = compute_nrel(ch, qdoc, qs, qe)
    valid = R >= 0; Rc = np.where(valid, R, 0)
    rel = valid & (ch.doc[Rc] == qdoc[:, None]) & (ch.cs[Rc] <= qs[:, None]) & (ch.ce[Rc] >= qe[:, None])
    out = {}
    for k in (1, 3, 5, 10): out[f"r{k}"] = rel[:, :k].any(1).astype(np.int8)
    rel10 = rel[:, :10]; has = rel10.any(1)
    out["mrr10"] = np.where(has, 1.0 / (rel10.argmax(1) + 1), 0.0)
    disc = 1.0 / np.log2(np.arange(2, 12))
    dcg = (rel10 * disc[:rel10.shape[1]]).sum(1)
    idcg = np.concatenate([[0.0], np.cumsum(disc)])[np.minimum(nrel, 10)]
    out["ndcg10"] = np.where(idcg > 0, dcg / np.where(idcg > 0, idcg, 1.0), 0.0)
    cwr = np.where(valid, ch.cw[Rc], 0); cum = np.cumsum(cwr, 1)
    for B in BUDGETS:
        out[f"bm{B}"] = (rel & valid & (cum <= B)).any(1).astype(np.int8)          # strict: never exceed budget
        out[f"bm{B}x"] = (rel & valid & ((cum - cwr) < B)).any(1).astype(np.int8)  # crossing: last chunk may overshoot
    out["split"] = (nrel == 0).astype(np.int8)
    out["words_used"] = np.where(valid & (cum <= 500), cum, 0).max(1).astype(np.int32)
    out["first_rel_rank"] = np.where(rel.any(1), rel.argmax(1) + 1, 0).astype(np.int16)
    return out

METRIC_KEYS = ["r1", "r3", "r5", "r10", "mrr10", "ndcg10", "bm200", "bm500", "bm1000", "bm2000", "bm200x", "bm500x", "bm1000x", "bm2000x"]
def summarize(pq, n):
    m = {"questions": int(n)}
    for k in METRIC_KEYS: m[k] = float(np.mean(pq[k])) if n else 0.0
    m["answer_split_rate"] = float(np.mean(pq["split"])) if n else 0.0
    return m

In [ ]:
# ============================== 6. RETRIEVERS (BM25 on sparse matrices, dense, RRF) ==============================
def topk_rows(S, k):
    k = min(k, S.shape[1])
    idx = np.argpartition(-S, k - 1, axis=1)[:, :k]
    sc = np.take_along_axis(S, idx, 1)
    order = np.lexsort((idx, -sc), axis=-1)
    return np.take_along_axis(idx, order, 1).astype(np.int32)

def chunk_matrix(corp, variant, ch):
    pos_type, T, _ = corp.sparse_index(variant)
    L = ch.gwj - ch.gwi; nc = len(L)
    offs = np.concatenate([[0], np.cumsum(L)[:-1]])
    pos = np.arange(int(L.sum())) + np.repeat(ch.gwi - offs, L)
    rows = np.repeat(np.arange(nc), L)
    CT = sp.csr_matrix((np.ones(len(pos), np.float32), (rows, pos_type[pos])), shape=(nc, T.shape[0]))
    return (CT @ T).tocsr()

def query_matrix(corp, variant, texts):
    _, T, vocab = corp.sparse_index(variant)
    r, c = [], []
    for i, t in enumerate(texts):
        for w in t.split():
            for term in term_list(clean_tok(w), variant):
                j = vocab.get(term)
                if j is not None: r.append(i); c.append(j)
    return sp.csr_matrix((np.ones(len(r), np.float32), (r, c)), shape=(len(texts), len(vocab)))

def bm25_weights(C, k1=1.2, b=0.75, eps=0.25):
    """BM25Okapi formula (identical to rank_bm25: idf = ln(N-df+.5)-ln(df+.5), negatives -> eps*avg_idf)."""
    C = C.tocsr().astype(np.float32); N = C.shape[0]
    dl = np.asarray(C.sum(1)).ravel(); avg = max(dl.mean(), 1e-9)
    df = np.bincount(C.indices, minlength=C.shape[1])
    present = df > 0
    idf = np.zeros(C.shape[1], np.float64)
    idf[present] = np.log(N - df[present] + 0.5) - np.log(df[present] + 0.5)
    avg_idf = idf[present].mean() if present.any() else 0.0
    idf[present & (idf < 0)] = eps * avg_idf
    rows = np.repeat(np.arange(N), np.diff(C.indptr))
    tf = C.data
    data = idf[C.indices] * tf * (k1 + 1) / (tf + k1 * (1 - b + b * dl[rows] / avg))
    return sp.csr_matrix((data.astype(np.float32), C.indices, C.indptr), shape=C.shape)

def bm25_rank(C, Q, k=100, bs=256):
    W = bm25_weights(C); Wt = W.T.tocsr()
    k = min(k, C.shape[0]); R = np.zeros((Q.shape[0], k), np.int32)
    for i in range(0, Q.shape[0], bs):
        S = (Q[i:i + bs] @ Wt).toarray().astype(np.float32)
        R[i:i + bs] = topk_rows(S, k)
    return R

def rrf_fuse(A, B, k=60, K=100):
    out = np.full((A.shape[0], K), -1, np.int32)
    ra = 1.0 / (k + np.arange(1, A.shape[1] + 1)); rb = 1.0 / (k + np.arange(1, B.shape[1] + 1))
    sc = np.concatenate([ra, rb])
    for i in range(A.shape[0]):
        ids = np.concatenate([A[i], B[i]])
        u, inv = np.unique(ids, return_inverse=True)
        tot = np.bincount(inv, weights=sc, minlength=len(u))
        o = np.lexsort((u, -tot))[:K]
        out[i, :len(o)] = u[o]
    return out

# ---- dense ----
def load_model(mk):
    from sentence_transformers import SentenceTransformer
    sp_ = MODEL_SPECS[mk]
    kw = dict(device="cuda", model_kwargs={"torch_dtype": torch.float16})
    if sp_["tok_kwargs"]: kw["tokenizer_kwargs"] = sp_["tok_kwargs"]
    m = SentenceTransformer(sp_["hf"], **kw)
    m.max_seq_length = sp_["max_seq"]
    m.tokenizer.model_max_length = 10**9
    TOKS[mk] = m.tokenizer
    MODEL_REV[mk] = model_revision(sp_["hf"])
    log(f"loaded {mk}: {sp_['hf']} rev={MODEL_REV[mk]} dtype=fp16 max_seq={sp_['max_seq']}")
    return m

def st_encode(model, texts, bs):
    tries = 0
    while True:
        try:
            return model.encode(texts, batch_size=bs, convert_to_tensor=True, normalize_embeddings=True, show_progress_bar=False)
        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache(); tries += 1
            if bs == 1 or tries > 6: raise
            bs = max(1, bs // 2); log("CUDA OOM, retrying with batch", bs)

def token_lengths(tok, texts, bs=2048):
    out = []
    for i in range(0, len(texts), bs):
        ids = tok(texts[i:i + bs], add_special_tokens=False, truncation=False, return_attention_mask=False)["input_ids"]
        out.extend(len(x) for x in ids)
    return np.array(out, np.int64)

def embed_queries(model, mk, texts, use_prefix=True):
    return st_encode(model, [fmt_query(mk, t, use_prefix) for t in texts], 64)

def dense_rank(Qe, Ce, k=100, bs=1024):
    Cf = Ce.float(); k = min(k, Cf.shape[0]); R = np.zeros((Qe.shape[0], k), np.int32)
    for i in range(0, Qe.shape[0], bs):
        s = Qe[i:i + bs].float() @ Cf.T
        R[i:i + bs] = torch.topk(s, k, dim=1).indices.cpu().numpy()
    return R

In [ ]:
# ============================== 7. RUN RECORDS, EMIT, RESUME ==============================
def build_and_emit(stage, rid, corp, split, retr, mk, cfg, ch, R, qsel, extra, cache_top100=False, base_runs=None):
    qv = qsel[corp.qvalid[qsel]]
    pq = score_rankings(ch, corp.qdoc[qv], corp.qs[qv], corp.qe[qv], R)
    metrics = summarize(pq, len(qv))
    n = len(qsel); pos = np.searchsorted(qsel, qv)
    def full(a, fill, dt):
        o = np.full(n, fill, dt); o[pos] = a; return o
    per = {}
    for k in ["r1", "r3", "r5", "r10"]: per[k] = full(pq[k], -1, np.int8).tolist()
    for k in ["mrr10", "ndcg10"]: per[k] = np.round(full(pq[k], -1.0, np.float64), 6).tolist()
    for k in ["bm200", "bm500", "bm1000", "bm2000", "bm200x", "bm500x", "bm1000x", "bm2000x", "split"]: per[k] = full(pq[k], -1, np.int8).tolist()
    per["words_used"] = full(pq["words_used"], -1, np.int32).tolist()
    per["first_rel_rank"] = full(pq["first_rel_rank"], -1, np.int16).tolist()
    rec = {
        "run_id": rid, "corpus": CORPUS_NAME[corp.tag], "split": split, "retriever": retr,
        "model": MODEL_SPECS[mk]["hf"] if mk in MODEL_SPECS else None,
        "model_revision": MODEL_REV.get(mk) if mk in MODEL_SPECS else None,
        "strategy": cfg.strategy, "size": cfg.size, "overlap": cfg.ov, "token_budget": cfg.budget,
        "normalization": corp.norm, "documents": int(len(np.unique(ch.doc))), "chunks": int(len(ch)),
        "truncated_chunks": 0, "truncation_rate": 0.0, "truncated": False,
        "embedding_sec": 0.0, "index_bytes": 0,
    }
    rec.update(extra or {})
    if base_runs: rec["base_runs"] = base_runs
    rec["chunk_words_mean"] = float(ch.cw.mean()) if len(ch) else 0.0
    rec["metrics"] = metrics
    if len(qv) < n:
        rec["excluded_question_positions"] = np.setdiff1d(np.arange(n), pos).tolist()
    rec["per_question"] = per
    top10 = np.full((n, 10), -1, np.int32); k10 = min(10, R.shape[1]); top10[pos, :k10] = R[:, :k10]
    pdir = WORK / "parts" / stage; pdir.mkdir(parents=True, exist_ok=True)
    np.savez_compressed(pdir / f"{rid}.npz", top10=top10, spans=ch.spans())
    if cache_top100:
        R16 = R.astype(np.uint16) if R.max() < 65535 else R.astype(np.int32)
        np.save(WORK / "cache" / f"top100_{rid}.npy", R16)
    append_line(RUNS_JSONL, dumps(rec))
    idx = {k: v for k, v in rec.items() if k != "per_question"}; idx["stage"] = stage
    append_line(INDEX_JSONL, dumps(idx))
    INDEX[rid] = json.loads(dumps(idx)); DONE.add(rid); save_state()
    return rec

## Stage A: data download, corpus construction, validation

In [ ]:
# ============================== STAGE A ==============================
SQUAD_URL = "https://raw.githubusercontent.com/sameearif/UQA/main/SQuAD/train-v2.0.json"
UQA_URL = "https://huggingface.co/datasets/uqa/UQA/resolve/main/data/train-00000-of-00001-bac007e8ca719235.parquet"
EXPECTED = {"uqa_rows": 124745, "english_paragraphs": 19035, "retained_rows": 24898, "eligible_articles": 403,
            "pool_articles": 150, "pool_dev": 30, "pool_test": 120, "pool_words": 1007122, "pool_paragraphs": 6720,
            "pool_questions": 9077, "dev_questions": 1715, "test_questions": 7362}
ACTUAL, HASHES, DATA_STATS = {}, {}, {}

def download(url, dest):
    if not dest.exists() or dest.stat().st_size == 0:
        log("downloading", url); urllib.request.urlretrieve(url, dest)
    return sha256_file(dest)

def first(x):
    if isinstance(x, (list, tuple, np.ndarray)): return x[0] if len(x) else None
    return x

def build_corpus():
    HASHES["squad_train_v2"] = download(SQUAD_URL, RAW / "train-v2.0.json")
    HASHES["uqa_train_parquet"] = download(UQA_URL, RAW / "uqa_train.parquet")
    log("sha256", HASHES)
    sq_data = json.load(open(RAW / "train-v2.0.json", encoding="utf-8"))["data"]
    sq, en_para = {}, {}
    for art in sq_data:
        t = art["title"]
        for pi, p in enumerate(art["paragraphs"]):
            en_para[(t, pi)] = p["context"]
            for qa in p["qas"]:
                sq[qa["id"]] = dict(title=t, pidx=pi, question=qa["question"], impossible=qa.get("is_impossible", False), answers=qa["answers"])
    df = pd.read_parquet(RAW / "uqa_train.parquet")
    ACTUAL["uqa_rows"] = len(df); ACTUAL["english_paragraphs"] = len(en_para)
    missing = [i for i in df["id"] if i not in sq]
    assert len(missing) == 0, f"{len(missing)} UQA ids unmatched to SQuAD"
    df["en_title"] = [sq[i]["title"] for i in df["id"]]; df["pidx"] = [sq[i]["pidx"] for i in df["id"]]
    # modal Urdu variant per English paragraph
    cnt = df.groupby(["en_title", "pidx", "context"], sort=False).size()
    best, nvar = {}, Counter()
    for (t, p, c), n in cnt.items():
        nvar[(t, p)] += 1; k = (n, len(c), c)
        if (t, p) not in best or k > best[(t, p)][0]: best[(t, p)] = (k, c)
    chosen = {k: v[1] for k, v in best.items()}
    ACTUAL["paragraphs_covered_by_uqa"] = len(chosen)
    modal_rows = sum(v[0][0] for v in best.values())
    DATA_STATS.update(mean_variants_per_paragraph=float(np.mean(list(nvar.values()))),
                      share_paragraphs_multi_variant=float(np.mean([v > 1 for v in nvar.values()])),
                      share_rows_in_modal_variant=modal_rows / len(df))
    print("variant duplication:", DATA_STATS)
    # retain rows
    retained, seen, dup = [], set(), 0
    for r in df.itertuples(index=False):
        if bool(r.is_impossible): continue
        if r.context != chosen[(r.en_title, r.pidx)]: continue
        if r.id in seen: dup += 1; continue
        seen.add(r.id)
        ans, st = first(r.answer), int(first(r.answer_start))
        retained.append(dict(qid=r.id, title=r.en_title, pidx=int(r.pidx), question=r.question, answer=ans, s=st, ctx=r.context))
    if dup: deviation(f"{dup} duplicate retained ids dropped (kept first)")
    ACTUAL["retained_rows"] = len(retained)
    bad = [x["qid"] for x in retained if x["ctx"][x["s"]:x["s"] + len(x["answer"])] != x["answer"]]
    DATA_STATS["retained_rate"] = len(retained) / len(df); DATA_STATS["span_mismatch_ur"] = len(bad)
    assert not bad, f"Urdu span audit failed for {len(bad)} rows, e.g. {bad[:3]}"
    # articles
    by_title = defaultdict(list)
    for x in retained: by_title[x["title"]].append(x)
    pids_by_title = defaultdict(list)
    for (t, p) in chosen: pids_by_title[t].append(p)
    arts = {}
    for t, pids in pids_by_title.items():
        pids = sorted(pids); paras_ur = [chosen[(t, p)] for p in pids]
        doc_ur = "\n\n".join(paras_ur); qs = by_title.get(t, [])
        arts[t] = dict(title=t, pids=pids, paras_ur=paras_ur, paras_en=[en_para[(t, p)] for p in pids], doc_ur=doc_ur,
                       words=len(doc_ur.split()), qs=qs)
    eligible = [t for t, a in arts.items() if len(a["qs"]) >= 20 and len(a["pids"]) >= 20 and a["words"] >= 1500]
    ACTUAL["eligible_articles"] = len(eligible)
    pool = sorted(eligible, key=lambda t: hashlib.sha1(t.encode()).hexdigest())[:150]
    # dev phase: "every 5th article" is ambiguous (positions 0,5,.. or 4,9,..); pick the phase matching the expected dev size
    def devq(ph): return sum(len(arts[t]["qs"]) for i, t in enumerate(pool) if i % 5 == ph)
    phase = min(range(5), key=lambda ph: (abs(devq(ph) - EXPECTED["dev_questions"]), 0 if ph == 4 else 1))
    print("dev question counts by phase:", {ph: devq(ph) for ph in range(5)}, "-> phase", phase)
    if phase != 4: deviation(f"'every 5th article is dev' is ambiguous; used positions i%5=={phase} because it matches the expected dev size")
    DATA_STATS["dev_phase"] = phase
    docs_ur, docs_en = [], []
    for i, t in enumerate(pool):
        a = arts[t]; split = "dev" if i % 5 == phase else "test"
        offs, pos = [], 0
        for p in a["paras_ur"]: offs.append(pos); pos += len(p) + 2
        offs_en, pos = [], 0
        for p in a["paras_en"]: offs_en.append(pos); pos += len(p) + 2
        doc_en = "\n\n".join(a["paras_en"]); rank = {p: k for k, p in enumerate(a["pids"])}
        qu, qe_ = [], []
        for x in a["qs"]:
            k = rank[x["pidx"]]; s = offs[k] + x["s"]; e = s + len(x["answer"])
            qu.append(dict(qid=x["qid"], question=x["question"], answer=x["answer"], s=s, e=e, para=x["pidx"]))
            sqq = sq[x["qid"]]; ea = sqq["answers"][0]; s2 = offs_en[k] + int(ea["answer_start"]); e2 = s2 + len(ea["text"])
            assert a["paras_en"][k][int(ea["answer_start"]):int(ea["answer_start"]) + len(ea["text"])] == ea["text"], f"English span mismatch {x['qid']}"
            qe_.append(dict(qid=x["qid"], question=sqq["question"], answer=ea["text"], s=s2, e=e2, para=x["pidx"]))
        assert all(a["doc_ur"][q["s"]:q["e"]] == q["answer"] for q in qu), "doc-level Urdu offsets wrong"
        assert all(doc_en[q["s"]:q["e"]] == q["answer"] for q in qe_), "doc-level English offsets wrong"
        docs_ur.append(dict(aid=i, title=t, split=split, doc=a["doc_ur"], paras=a["paras_ur"], qs=qu))
        docs_en.append(dict(aid=i, title=t, split=split, doc=doc_en, paras=a["paras_en"], qs=qe_))
    ACTUAL.update(pool_articles=len(pool), pool_dev=sum(d["split"] == "dev" for d in docs_ur), pool_test=sum(d["split"] == "test" for d in docs_ur),
                  pool_words=sum(len(d["doc"].split()) for d in docs_ur), pool_paragraphs=sum(len(d["paras"]) for d in docs_ur),
                  pool_questions=sum(len(d["qs"]) for d in docs_ur),
                  dev_questions=sum(len(d["qs"]) for d in docs_ur if d["split"] == "dev"),
                  test_questions=sum(len(d["qs"]) for d in docs_ur if d["split"] == "test"))
    n_nfc = sum(not unicodedata.is_normalized("NFC", d["doc"]) for d in docs_ur)
    DATA_STATS["docs_not_nfc"] = n_nfc
    print("documents not in NFC (reported, NOT normalized):", n_nfc, "/", len(docs_ur))
    return docs_ur, docs_en

def check_counts():
    rows, ok_all = [], True
    for k, exp in EXPECTED.items():
        act = ACTUAL.get(k); ok = act is not None and abs(act - exp) <= 0.005 * exp
        ok_all &= ok; rows.append((k, exp, act, "ok" if ok else "MISMATCH"))
    print(pd.DataFrame(rows, columns=["quantity", "expected", "actual", "status"]).to_string(index=False))
    print("also: english paragraphs covered by UQA =", ACTUAL.get("paragraphs_covered_by_uqa"), "(expected 19,021)")
    return ok_all

docs_ur, docs_en = build_corpus()
counts_ok = check_counts()
report_test("1 counts within 0.5% of spec", counts_ok, json.dumps({k: ACTUAL[k] for k in ("pool_articles", "pool_dev", "pool_test", "pool_questions", "pool_words")}))
report_test("2 span audit (Urdu + English) exact", True, f"{ACTUAL['retained_rows']} Urdu rows and all English spans matched exactly (asserted)")
if STRICT_COUNTS and not counts_ok and not DRY_RUN:
    raise AssertionError("Corpus counts deviate from the spec by more than 0.5%. Stop and report (see table above).")

# restrict to 2 articles for the dry run (after validating on the full pool)
if DRY_RUN:
    pick = [next(i for i, d in enumerate(docs_ur) if d["split"] == "dev"), next(i for i, d in enumerate(docs_ur) if d["split"] == "test")]
    docs_ur = [docs_ur[i] for i in pick]; docs_en = [docs_en[i] for i in pick]
    for j, (a, b) in enumerate(zip(docs_ur, docs_en)): a["aid"] = b["aid"] = j
    log("DRY_RUN: pool reduced to", [d["title"] for d in docs_ur], [d["split"] for d in docs_ur])
BASE["ur"] = dict(tag="ur", lang="ur", docs=docs_ur)
BASE["en"] = dict(tag="en", lang="en", docs=docs_en)

# ---- native set (optional) ----
def _find(name):
    for d in (WORK, WORK.parent, Path("/content"), Path("/content/drive/MyDrive")):
        if (d / name).exists(): return d / name
    return None

def build_native():
    f1, f2 = _find("native_validated.jsonl"), _find("native_articles.json")
    if not f1 or not f2:
        log("native files absent -> stage F skipped"); return None
    arts = json.load(open(f2, encoding="utf-8"))
    if isinstance(arts, dict): arts = arts.get("articles", list(arts.values()))
    keys, docs = {}, []
    for i, a in enumerate(arts):
        paras = a["paragraphs"]; doc = "\n\n".join(paras); offs, pos = [], 0
        for p in paras: offs.append(pos); pos += len(p) + 2
        docs.append(dict(aid=i, title=a["title"], split="all", doc=doc, paras=paras, offs=offs, qs=[]))
        for k in (a.get("pageid"), a.get("title"), a.get("url"), a.get("revid")):
            if k is not None: keys[str(k)] = i
    stats = Counter()
    for line in open(f1, encoding="utf-8"):
        if not line.strip(): continue
        it = json.loads(line); stats["items"] += 1
        di = keys.get(str(it.get("article_id")), keys.get(str(it.get("article_title"))))
        if di is None: stats["unmatched_article"] += 1; continue
        d = docs[di]; pi = it.get("paragraph_index"); s, e, ans = int(it["answer_start"]), int(it["answer_end"]), it["answer"]
        cand = []
        if pi is not None and 0 <= int(pi) < len(d["offs"]): cand.append(("paragraph", d["offs"][int(pi)] + s, d["offs"][int(pi)] + e))
        cand.append(("document", s, e))
        hit = next(((m, a, b) for m, a, b in cand if d["doc"][a:b] == ans), None)
        if hit is None: stats["span_failed"] += 1; continue
        stats["span_" + hit[0]] += 1
        d["qs"].append(dict(qid=str(it["item_id"]), question=it["question"], answer=ans, s=hit[1], e=hit[2], para=pi))
    print("native stats:", dict(stats)); DATA_STATS["native"] = dict(stats)
    if stats.get("span_failed") or stats.get("unmatched_article"):
        deviation(f"native: {stats.get('span_failed',0)} items failed span check, {stats.get('unmatched_article',0)} unmatched articles (excluded)")
    return dict(tag="nat", lang="ur", docs=docs)

NATIVE = build_native() if (("F" in STAGES) and not DRY_RUN) else None
if NATIVE: BASE["nat"] = NATIVE

# ---- write corpora, questions.csv, audit file ----
def write_corpus_files():
    for tag, name in (("ur", "corpus_ur.jsonl"), ("en", "corpus_en.jsonl")):
        with open(WORK / name, "w", encoding="utf-8") as f:
            for d in BASE[tag]["docs"]:
                f.write(dumps({"article_id": d["aid"], "split": d["split"], "title": d["title"],
                               "paragraphs": [{"pid": i, "text": t} for i, t in enumerate(d["paras"])], "doc": d["doc"],
                               "questions": [{"qid": q["qid"], "question": q["question"], "answer": q["answer"], "start": q["s"], "end": q["e"], "paragraph": q["para"]} for q in d["qs"]]}) + "\n")
    rows = []
    for du, de in zip(BASE["ur"]["docs"], BASE["en"]["docs"]):
        for qu, qe_ in zip(du["qs"], de["qs"]):
            rows.append(dict(qid=qu["qid"], article_id=du["aid"], split=du["split"], question=qu["question"], answer=qu["answer"],
                             answer_start=qu["s"], answer_end=qu["e"], paragraph=qu["para"], question_en=qe_["question"],
                             answer_en=qe_["answer"], answer_start_en=qe_["s"], answer_end_en=qe_["e"]))
    pd.DataFrame(rows).to_csv(WORK / "questions.csv", index=False, encoding="utf-8")
    if NATIVE:
        nr = [dict(qid=q["qid"], article_id=d["aid"], split="all", question=q["question"], answer=q["answer"], answer_start=q["s"], answer_end=q["e"]) for d in NATIVE["docs"] for q in d["qs"]]
        pd.DataFrame(nr).to_csv(WORK / "questions_native.csv", index=False, encoding="utf-8")
    dev = [(d["title"], q) for d in BASE["ur"]["docs"] if d["split"] == "dev" for q in d["qs"]]
    pick = random.Random(0).sample(dev, min(100, len(dev)))
    pd.DataFrame([dict(qid=q["qid"], article_title=t, question=q["question"], gold_answer=q["answer"], self_contained="") for t, q in pick]).to_csv(WORK / "audit_100_questions.csv", index=False, encoding="utf-8")
write_corpus_files()
log("corpus files, questions.csv and audit_100_questions.csv written")

# ---- offset mapping test (N0/N1/N2) ----
NORM_STATS = {}
for _n in ("N0", "N1", "N2"):
    _c = get_corpus("ur", _n)
    NORM_STATS[_n] = dict(questions=_c.n_total_q, failed=_c.fail_q, failure_rate=_c.fail_q / max(1, _c.n_total_q))
report_test("4 offset mapping N0/N1/N2", all(v["failed"] == 0 for v in NORM_STATS.values()),
            "; ".join(f"{k}: {v['failed']}/{v['questions']} failed ({v['failure_rate']:.4%})" for k, v in NORM_STATS.items()) + " (failed questions are excluded from that variant, never re-found)")

# ---- chunker coverage test ----
def all_test_cfgs():
    out = set()
    for c in bm25_grid(): out.add((c, BM25_TOKENIZER_KEY))
    for mk in set(MODELS) | {"bge", "qwen"}:
        for c in dense_core_grid() + dense_ext_grid() + labse_grid():
            out.add((c, mk if c.strategy == "fixed_tokens" else None))
    return sorted(out, key=lambda x: (x[0].strategy, x[0].size or 0, x[0].budget or 0, x[0].ov, str(x[1])))

def coverage_test():
    corp = get_corpus("ur", "N0"); ids = list(range(min(10, len(corp.docs))))
    NS = {di: np.fromiter((not c.isspace() for c in corp.docs[di]), bool, len(corp.docs[di])) for di in ids}
    worst_unc, worst_dbl, n = 0, 0, 0
    for cfg, tk in all_test_cfgs():
        if cfg.strategy == "fixed_tokens" and tk is None: continue
        if cfg.strategy == "fixed_tokens" and tk not in MODEL_SPECS: continue
        for di in ids:
            doc = corp.docs[di]; ch = make_chunks(corp, [di], cfg, tk)
            diff = np.zeros(len(doc) + 1, np.int32); np.add.at(diff, ch.cs, 1); np.add.at(diff, ch.ce, -1)
            cov = np.cumsum(diff)[:-1]; ns = NS[di]
            unc = int((ns & (cov == 0)).sum()); dbl = int((ns & (cov > 1)).sum()) if cfg.ov == 0 else 0
            worst_unc = max(worst_unc, unc); worst_dbl = max(worst_dbl, dbl); n += 1
    return worst_unc, worst_dbl, n

_t = time.time(); _u, _d, _n = coverage_test()
report_test("3 chunker coverage (10 docs x all configs)", _u == 0 and _d == 0, f"max uncovered={_u}, max double-covered at overlap 0={_d}, checks={_n}, {time.time()-_t:.0f}s")
log("Stage A complete")

In [ ]:
# ============================== SELF-TESTS 5, 6 (no GPU) ==============================
def test_oracle_random():
    rng = np.random.default_rng(0); nc, nq = 200, 4000
    cs = np.arange(nc) * 100; ch = ChunkSet(np.zeros(nc, np.int64), cs, cs + 99, np.full(nc, 10), np.zeros(nc, np.int64), np.zeros(nc, np.int64))
    g = rng.integers(0, nc, nq); qs = cs[g] + 10; qe = cs[g] + 20; qd = np.zeros(nq, np.int64)
    Ror = np.stack([np.concatenate([[x], rng.permutation(np.delete(np.arange(nc), x))[:99]]) for x in g])
    Rra = np.stack([rng.permutation(nc)[:100] for _ in range(nq)])
    o = score_rankings(ch, qd, qs, qe, Ror); r = score_rankings(ch, qd, qs, qe, Rra)
    ok = o["bm500"].mean() == 1.0 and o["r1"].mean() == 1.0 and abs(o["ndcg10"].mean() - 1) < 1e-9 and abs(r["r5"].mean() - 5 / 200) < 0.01
    report_test("5 oracle / random sanity", ok, f"oracle bm500={o['bm500'].mean():.3f}; random r5={r['r5'].mean():.4f} (expect {5/200:.4f})")

def test_bm25_vs_reference():
    corp = get_corpus("ur", "N0"); split = "dev" if len(corp.split_docs("dev")) else "test"
    cfg = Cfg("fixed_words", 100, 0.0, None); ch = make_chunks(corp, corp.split_docs(split), cfg)
    qsel = corp.split_q(split); qsel = qsel[corp.qvalid[qsel]]; qsel = qsel[:25]
    C = chunk_matrix(corp, "ws", ch); Q = query_matrix(corp, "ws", [corp.q_text[i] for i in qsel])
    mine = (Q @ bm25_weights(C).T.tocsr()).toarray()
    toks = [[t for t in (clean_tok(w) for w in corp.docs[d][s:e].split()) if t] for d, s, e in zip(ch.doc, ch.cs, ch.ce)]
    try:
        from rank_bm25 import BM25Okapi
        bm = BM25Okapi(toks, k1=1.2, b=0.75); ref_name = "rank_bm25.BM25Okapi"
        ref = np.stack([bm.get_scores([t for t in (clean_tok(w) for w in corp.q_text[i].split()) if t]) for i in qsel])
    except Exception as ex:
        ref_name = f"independent Counter-based implementation ({ex!r})"
        N = len(toks); df = Counter(t for tt in toks for t in set(tt)); avg = np.mean([len(t) for t in toks]); tfs = [Counter(t) for t in toks]
        idf = {t: math.log(N - n + 0.5) - math.log(n + 0.5) for t, n in df.items()}
        aidf = np.mean(list(idf.values())); idf = {t: (v if v >= 0 else 0.25 * aidf) for t, v in idf.items()}
        ref = np.zeros_like(mine)
        for qi, i in enumerate(qsel):
            for t in [t for t in (clean_tok(w) for w in corp.q_text[i].split()) if t]:
                if t not in idf: continue
                for ci, tf in enumerate(tfs):
                    f = tf.get(t, 0)
                    if f: ref[qi, ci] += idf[t] * f * 2.2 / (f + 1.2 * (0.25 + 0.75 * len(toks[ci]) / avg))
    rhos = [spearmanr(mine[i], ref[i]).correlation for i in range(len(qsel)) if np.std(ref[i]) > 0]
    report_test("6 BM25 vs reference (Spearman on chunk scores)", float(np.mean(rhos)) > 0.99, f"vs {ref_name}: mean rho={np.mean(rhos):.5f}, min={np.min(rhos):.5f}, queries={len(rhos)}")

test_oracle_random(); test_bm25_vs_reference()

## Stage B: BM25 (CPU)

In [ ]:
# ============================== STAGE B ==============================
def run_bm25(stage, corp, split, variant, cfg, tkey):
    retr = "bm25_ws" if variant == "ws" else "bm25_char"
    rid = make_run_id(corp.tag, split, "none", cfg, corp.norm, retr)
    if rid in DONE: return None
    t0 = time.time()
    ch = make_chunks(corp, corp.split_docs(split), cfg, tkey)
    qsel = corp.split_q(split); qv = qsel[corp.qvalid[qsel]]
    C = chunk_matrix(corp, variant, ch); Q = query_matrix(corp, variant, [corp.q_text[i] for i in qv])
    R = bm25_rank(C, Q, 100)
    rec = build_and_emit(stage, rid, corp, split, retr, "none", cfg, ch, R, qsel,
                         extra={"index_bytes": int(C.data.nbytes + C.indices.nbytes + C.indptr.nbytes), "embedding_sec": 0.0},
                         cache_top100=(corp.norm == "N0" and variant == "ws" and corp.tag == "ur"))
    return time.time() - t0, rec

def stage_B():
    log("Stage B: BM25")
    corp = get_corpus("ur", "N0"); char_times, reduced = [], False
    for variant in ("ws", "char"):
        grid = bm25_grid(); total = len(grid) * len(RUN_SPLITS)
        for split in RUN_SPLITS:
            for i, cfg in enumerate(grid):
                if variant == "char" and reduced and cfg.ov > 0: continue
                out = run_bm25("B", corp, split, variant, cfg, BM25_TOKENIZER_KEY)
                if out is None: continue
                dt, rec = out
                if i % 10 == 0: log(f"  BM25-{variant} {split} {i+1}/{len(grid)} {cfg} bm500={rec['metrics']['bm500']:.4f} ({dt:.0f}s)")
                if variant == "char":
                    char_times.append(dt)
                    if CHAR_GRID_MODE == "reduced" and not reduced: reduced = True
                    if CHAR_GRID_MODE == "auto" and len(char_times) == 6 and np.mean(char_times) * total > CHAR_TIME_BUDGET_SEC and not reduced:
                        reduced = True
                        deviation(f"char BM25 grid reduced to overlap 0 only (projected {np.mean(char_times)*total/3600:.1f} h > budget); whitespace BM25 grid is full")
    # normalization variants (N0 runs already exist in the full grid)
    for n in ("N1", "N2"):
        cn = get_corpus("ur", n)
        for variant in ("ws", "char"):
            for cfg in norm_grid():
                for split in RUN_SPLITS: run_bm25("B", cn, split, variant, cfg, BM25_TOKENIZER_KEY)
    log("Stage B done. mapping failure rates:", {k: f"{v['failure_rate']:.4%}" for k, v in NORM_STATS.items()})

if "B" in STAGES: stage_B()

## Stage C/D/F: dense retrieval (GPU)

In [ ]:
# ============================== DENSE HELPERS ==============================
def dense_selftests(mk, model, corp):
    tok = model.tokenizer; ms = MODEL_SPECS[mk]["max_seq"]
    ch = make_chunks(corp, corp.split_docs("test")[:1], Cfg("fixed_words", 100, 0.0, None))
    texts = [corp.docs[d][s:e] for d, s, e in list(zip(ch.doc, ch.cs, ch.ce))[:20]]
    a = st_encode(model, texts, 32).float().cpu().numpy(); b = st_encode(model, texts, 32).float().cpu().numpy()
    report_test(f"7 embedding determinism ({mk})", float(np.abs(a - b).max()) < 1e-5, f"max abs diff={np.abs(a-b).max():.2e}")
    long_t = " ".join(["سلام"] * (ms + 50)); short_t = "یہ ایک مختصر جملہ ہے"
    L = token_lengths(tok, [long_t, short_t])
    report_test(f"8 truncation flag ({mk})", L[0] > ms - 2 and L[1] <= ms - 2, f"long={L[0]} tokens (flagged), short={L[1]} (not flagged), limit={ms-2}")

def prefix_check(mk, model, corp):
    if MODEL_SPECS[mk]["prefix"] is None: return
    split = "dev" if len(corp.split_docs("dev")) else "test"
    qsel = corp.split_q(split); qsel = qsel[corp.qvalid[qsel]]
    qsel = np.array(sorted(random.Random(0).sample(list(qsel), min(100, len(qsel)))))
    ch = make_chunks(corp, corp.split_docs(split), Cfg("fixed_words", 200, 0.0, None))
    E = st_encode(model, [corp.docs[d][s:e] for d, s, e in zip(ch.doc, ch.cs, ch.ce)], 32)
    res = {}
    for use in (True, False):
        Qe = embed_queries(model, mk, [corp.q_text[i] for i in qsel], use)
        R = dense_rank(Qe, E, 100)
        m = summarize(score_rankings(ch, corp.qdoc[qsel], corp.qs[qsel], corp.qe[qsel], R), len(qsel))
        res["prefix" if use else "no_prefix"] = {k: round(m[k], 4) for k in ("r1", "r5", "r10", "bm500")}
    PREFIX_CHECK[mk] = res
    report_test(f"9 prefix check ({mk}, {split}, 200 words, n={len(qsel)})", True, json.dumps(res) + " | documented prefix is used for all runs regardless")

PREFIX_CHECK = {}
def save_aux():
    AUX_JSON.write_text(json.dumps({"tests": TESTS, "fertility": FERTILITY, "prefix": PREFIX_CHECK, "model_rev": MODEL_REV}, ensure_ascii=False, default=_np_default))

def load_aux():
    if AUX_JSON.exists():
        try: a = json.loads(AUX_JSON.read_text())
        except Exception: return
        for k, v in a.get("tests", {}).items(): TESTS.setdefault(k, v)
        for k, v in a.get("fertility", {}).items(): FERTILITY.setdefault(k, v)
        for k, v in a.get("prefix", {}).items(): PREFIX_CHECK.setdefault(k, v)
        for k, v in a.get("model_rev", {}).items(): MODEL_REV.setdefault(k, v)

def compute_fertility(mk, tok):
    out = {}
    for tag in ("ur", "en"):
        corp = get_corpus(tag, "N0"); n_tok = n_w = 0
        for di in range(min(30, len(corp.docs))):
            n_tok += len(tok(corp.docs[di], add_special_tokens=False, truncation=False)["input_ids"]); n_w += corp.feat(di).n
        out[tag] = n_tok / max(1, n_w)
    FERTILITY[mk] = out; log(f"tokens/word {mk}: {out}")

def run_dense_grid(stage, mk, model, corp, split, cfgs, qemb, retr="dense"):
    tok = model.tokenizer; ms = MODEL_SPECS[mk]["max_seq"]
    qsel = corp.split_q(split); qv = qsel[corp.qvalid[qsel]]; qv_t = torch.as_tensor(qv, device=qemb.device)
    for n, cfg in enumerate(cfgs):
        rid = make_run_id(corp.tag, split, mk, cfg, corp.norm, "dense")
        if rid in DONE: continue
        ch = make_chunks(corp, corp.split_docs(split), cfg, mk)
        texts = [corp.docs[d][s:e] for d, s, e in zip(ch.doc, ch.cs, ch.ce)]
        L = token_lengths(tok, texts); trunc = L > ms - 2; rate = float(trunc.mean()) if len(L) else 0.0
        t0 = time.time(); E = st_encode(model, texts, 32); torch.cuda.synchronize(); emb_sec = time.time() - t0
        R = dense_rank(qemb[qv_t], E, 100)
        extra = dict(truncated_chunks=int(trunc.sum()), truncation_rate=rate, truncated=bool(rate > 0.05), embedding_sec=round(emb_sec, 2),
                     index_bytes=int(E.shape[0] * E.shape[1] * 2), chunk_tokens_mean=float(L.mean()), chunk_tokens_p95=float(np.percentile(L, 95)))
        if cfg.strategy == "fixed_tokens": extra["budget_overflow_rate"] = float((L > cfg.budget).mean())
        rec = build_and_emit(stage, rid, corp, split, retr, mk, cfg, ch, R, qsel, extra, cache_top100=(corp.tag == "ur" and corp.norm == "N0" and cfg.strategy != "fixed_tokens"))
        log(f"  [{mk}/{corp.tag}/{split}] {n+1}/{len(cfgs)} {cfg.strategy} {cfg.size or cfg.budget}/{int(cfg.ov*100)} chunks={len(ch)} trunc={rate:.1%} "
            f"bm500={rec['metrics']['bm500']:.4f} r10={rec['metrics']['r10']:.4f} emb={emb_sec:.0f}s")
        if len(ch) / max(emb_sec, 1e-6) < 7000 / 60 / 10 and (cfg.size or cfg.budget) <= 200 and not DRY_RUN:
            raise RuntimeError(f"Embedding is >10x slower than the pilot ({len(ch)/emb_sec:.1f} chunks/s vs ~117). Stop and report.")
        del E; torch.cuda.empty_cache()

def free_model(model):
    del model; gc.collect(); torch.cuda.empty_cache()

In [ ]:
# ============================== STAGE C: dense Urdu ==============================
def grid_for_model(mk):
    if DRY_RUN: return [Cfg("fixed_words", 100, 0.0, None)]
    if mk == "labse": return labse_grid()
    return dense_core_grid() + (dense_ext_grid() if mk in ("bge", "qwen") else [])

def stage_C():
    assert torch.cuda.is_available(), "GPU required for stage C. Runtime > Change runtime type > GPU."
    corp = get_corpus("ur", "N0"); tests_done = False
    for mk in (MODELS[:1] if DRY_RUN else MODELS):
        grid = grid_for_model(mk)
        todo = [(sp_, c) for sp_ in RUN_SPLITS for c in grid if make_run_id("ur", sp_, mk, c, "N0", "dense") not in DONE]
        log(f"Stage C [{mk}]: {len(todo)} runs to do of {len(grid)*len(RUN_SPLITS)}")
        if not todo and not DRY_RUN:
            if mk not in FERTILITY: compute_fertility(mk, get_tokenizer(mk)); save_aux()
            continue
        model = load_model(mk)
        compute_fertility(mk, model.tokenizer)
        if not tests_done: dense_selftests(mk, model, corp); tests_done = True
        prefix_check(mk, model, corp)
        save_aux()
        if todo:
            qemb = embed_queries(model, mk, corp.q_text, True)
            for sp_ in RUN_SPLITS: run_dense_grid("C", mk, model, corp, sp_, grid, qemb)
            del qemb
        free_model(model)

load_aux()
if "C" in STAGES: stage_C()

In [ ]:
# ============================== STAGE D: English control (bge) ==============================
def stage_D():
    load_aux()
    if DRY_RUN or "bge" not in MODELS: log("Stage D skipped (dry run or bge not in MODELS)"); return
    corp = get_corpus("en", "N0"); grid = dense_core_grid()
    todo = [1 for sp_ in RUN_SPLITS for c in grid if make_run_id("en", sp_, "bge", c, "N0", "dense") not in DONE]
    log(f"Stage D: {len(todo)} runs to do")
    if not todo: return
    model = load_model("bge"); qemb = embed_queries(model, "bge", corp.q_text, True)
    for sp_ in RUN_SPLITS: run_dense_grid("D", "bge", model, corp, sp_, grid, qemb)
    free_model(model)

if "D" in STAGES: stage_D()

In [ ]:
# ============================== STAGE E: hybrid RRF (no embedding) ==============================
def stage_E():
    log("Stage E: RRF")
    corp = get_corpus("ur", "N0"); made = 0
    for rid, r in list(INDEX.items()):
        if not (r["corpus"] == "uqa_ur" and r["retriever"] == "dense" and r["normalization"] == "N0" and r["strategy"] != "fixed_tokens"): continue
        cfg = Cfg(r["strategy"], r["size"], r["overlap"], None); split = r["split"]
        mk = next(k for k, v in MODEL_SPECS.items() if v["hf"] == r["model"])
        bid = make_run_id("ur", split, "none", cfg, "N0", "bm25_ws"); out_id = make_run_id("ur", split, mk, cfg, "N0", "rrf")
        if out_id in DONE or bid not in INDEX: continue
        pa, pb = WORK / "cache" / f"top100_{rid}.npy", WORK / "cache" / f"top100_{bid}.npy"
        if not (pa.exists() and pb.exists()): log("  missing top100 cache for", out_id); continue
        ch = make_chunks(corp, corp.split_docs(split), cfg)
        assert len(ch) == r["chunks"] == INDEX[bid]["chunks"], f"chunk-set mismatch for {out_id}"
        A, B = np.load(pa).astype(np.int32), np.load(pb).astype(np.int32)
        R = rrf_fuse(A, B)
        qsel = corp.split_q(split)
        assert len(qsel[corp.qvalid[qsel]]) == len(R)
        rec = build_and_emit("E", out_id, corp, split, "rrf", mk, cfg, ch, R, qsel, extra={}, base_runs=[rid, bid])
        made += 1
        if made % 10 == 1: log(f"  {out_id} bm500={rec['metrics']['bm500']:.4f}")
    log(f"Stage E done: {made} new RRF runs")

if "E" in STAGES: stage_E()

In [ ]:
# ============================== STAGE F: native set ==============================
def stage_F():
    if DRY_RUN or "nat" not in BASE: log("Stage F skipped (no native files or dry run)"); return
    assert torch.cuda.is_available(), "GPU required"
    corp = get_corpus("nat", "N0"); grid = dense_core_grid()
    for mk in [m for m in ("bge", "qwen") if m in MODELS]:
        todo = [c for c in grid if make_run_id("nat", "all", mk, c, "N0", "dense") not in DONE]
        log(f"Stage F [{mk}]: {len(todo)} runs to do")
        if not todo: continue
        model = load_model(mk); qemb = embed_queries(model, mk, corp.q_text, True)
        run_dense_grid("F", mk, model, corp, "all", grid, qemb)
        free_model(model)

if "F" in STAGES: stage_F()

## Finalize: dedupe, rankings npz, manifest, README, summary JSON, examples, zip

In [ ]:
# ============================== FINALIZE ==============================
def dedupe_runs():
    if not RUNS_JSONL.exists(): return 0
    last, n = {}, 0
    for line in open(RUNS_JSONL, encoding="utf-8"):
        m = re.match(r'\{"run_id":"([^"]+)"', line)
        if m: last[m.group(1)] = line; n += 1
    with open(RUNS_JSONL, "w", encoding="utf-8") as f:
        for l in last.values(): f.write(l if l.endswith("\n") else l + "\n")
    return n - len(last)

def consolidate_rankings():
    for stage in "BCDEF":
        d = WORK / "parts" / stage
        if not d.exists(): continue
        files = sorted(d.glob("*.npz"))
        if not files: continue
        arrs = {}
        for p in files:
            z = np.load(p); arrs[p.stem + "|top10"] = z["top10"]; arrs[p.stem + "|spans"] = z["spans"]
        np.savez_compressed(WORK / f"rankings_{stage}.npz", run_ids=np.array([p.stem for p in files]), **arrs)
        log(f"rankings_{stage}.npz: {len(files)} runs")

def best_dense_model():
    c = [r for r in INDEX.values() if r["retriever"] == "dense" and r["corpus"] == "uqa_ur" and r["strategy"] == "fixed_words" and r["size"] == 100 and r["overlap"] == 0 and r["normalization"] == "N0" and r["split"] == "test"]
    return max(c, key=lambda r: r["metrics"]["bm500"])["model"] if c else None

def examples_text(n=5):
    hf = best_dense_model()
    if hf is None: return "no dense Urdu runs available for examples\n"
    mk = next(k for k, v in MODEL_SPECS.items() if v["hf"] == hf); corp = get_corpus("ur", "N0"); out = [f"Examples with dense model {mk}\n"]
    for strat in ("fixed_words", "sentence_aware", "recursive_english"):
        rid = make_run_id("ur", "test", mk, Cfg(strat, 100, 0.0, None), "N0", "dense")
        if rid not in INDEX: continue
        z = np.load(WORK / "parts" / INDEX[rid]["stage"] / f"{rid}.npz"); top10, spans = z["top10"], z["spans"]
        qsel = corp.split_q("test"); picks = random.Random(SEED).sample(range(len(qsel)), min(n, len(qsel)))
        out.append("=" * 100 + f"\nRUN {rid}\n")
        for pi in picks:
            qi = qsel[pi]; d, gs, ge = corp.qdoc[qi], corp.qs[qi], corp.qe[qi]; doc = corp.docs[d]
            gold = np.flatnonzero((spans[:, 0] == d) & (spans[:, 1] <= gs) & (spans[:, 2] >= ge))
            out.append(f"Q [{corp.q_id[qi]}]: {corp.q_text[qi]}\nGOLD ANSWER: {corp.q_ans[qi]}")
            out.append("GOLD CHUNK: " + (doc[spans[gold[0], 1]:spans[gold[0], 2]][:500] if len(gold) else "<<NO CHUNK CONTAINS THE FULL ANSWER (answer split)>>"))
            for rank, c in enumerate(top10[pi][:3], 1):
                if c < 0: continue
                dd, s, e = spans[c]; flag = int(dd == d and s <= gs and e >= ge)
                out.append(f"  TOP{rank} relevant={flag}: {corp.docs[dd][s:e][:500]}")
            out.append("-" * 100)
    return "\n".join(out) + "\n"

def build_summary():
    runs = [{"run_id": r["run_id"], "metrics": {k: (round(v, 5) if isinstance(v, float) else v) for k, v in r["metrics"].items()}} for r in INDEX.values()]
    nq = max((r["metrics"]["questions"] for r in INDEX.values()), default=0)
    s = dumps({"schema": "1.0", "n_questions": nq, "runs": runs})
    if len(s) > 190_000:
        runs = [{"run_id": r["run_id"], "metrics": {k: round(v, 4) for k, v in r["metrics"].items() if not k.endswith("x")}} for r in INDEX.values()]
        s = dumps({"schema": "1.0", "n_questions": nq, "runs": runs})
    return s

def write_readme():
    txt = """# Urdu chunking study: results package

Order of questions: rows of questions.csv with split == the run's split, in file order. Every per_question array and every
row of `top10` follows that order. Native runs use questions_native.csv (all rows).

- manifest.json: schema, timestamps, GPU/library versions, SHA-256 of inputs, model revisions, expected vs actual counts, self-test results, tokenizer fertility, prefix checks, settings, deviations.
- corpus_ur.jsonl / corpus_en.jsonl: one record per article (doc text, paragraphs, questions with doc-level offsets).
- questions.csv: Urdu question/answer/offsets plus English question_en/answer_en/offsets for the English control.
- runs.jsonl: one record per run with metrics and per_question arrays (schema in the task spec). Extras: `bm*x` = crossing-budget variant (last chunk may overshoot), `chunk_words_mean`, `chunk_tokens_mean/p95`, `budget_overflow_rate` (fixed_tokens), `excluded_question_positions` (N1/N2 mapping failures; their per_question entries are -1).
- run_index.jsonl: same as runs.jsonl without per_question (light).
- rankings_<stage>.npz: keys `<run_id>|top10` (int32 [n_questions,10] global chunk ids, -1 = none/excluded) and `<run_id>|spans` (int32 [n_chunks,3] = article_index, start, end in the normalized document of that run). `run_ids` lists all runs. Chunk ids index rows of that run's `spans`. article_index = article_id in corpus_*.jsonl.
- audit_100_questions.csv: 100 dev questions for the human self-containedness audit (fill `self_contained`).
- examples.txt: five full examples for three strategies (chunk-boundary inspection).
- summary.json: the SUMMARY_JSON printed at the end.
- tokenizer_fertility.json: tokens per word (Urdu / English) per model tokenizer.
- state.json: completed run ids.

Definitions: budget metrics are STRICT by default (cumulative chunk words must not exceed the budget; first chunk is not exempt). `bm*x` allows the chunk that crosses the budget. Relevance is article-scoped; retrieval searches the pooled chunks of the run's split.
"""
    (WORK / "RESULTS_README.md").write_text(txt, encoding="utf-8")

def write_manifest():
    gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else None
    import sentence_transformers, transformers
    man = {"schema_version": "1.0", "timestamp": time.strftime("%Y-%m-%dT%H:%M:%S"), "dry_run": DRY_RUN, "gpu": gpu, "torch": torch.__version__,
           "cuda": torch.version.cuda, "sentence_transformers": sentence_transformers.__version__, "transformers": transformers.__version__,
           "sha256": HASHES, "model_revisions": MODEL_REV, "expected_counts": EXPECTED, "actual_counts": ACTUAL, "data_stats": DATA_STATS,
           "normalization_mapping": NORM_STATS, "self_tests": TESTS, "tokenizer_fertility": FERTILITY, "prefix_checks": PREFIX_CHECK,
           "settings": dict(STAGES=STAGES, MODELS=MODELS, RUN_SPLITS=RUN_SPLITS, CHAR_GRID_MODE=CHAR_GRID_MODE, EN_BOUNDARIES_FOR_EN_CORPUS=EN_BOUNDARIES_FOR_EN_CORPUS,
                            budget_mode_primary="strict", bm25=dict(k1=1.2, b=0.75, idf="BM25Okapi"), rrf_k=60, bm25_fixed_tokens_tokenizer=BM25_TOKENIZER_KEY, seed=SEED),
           "n_runs": len(INDEX), "deviations_runtime": DEVIATIONS}
    (WORK / "manifest.json").write_text(json.dumps(man, ensure_ascii=False, indent=1, default=_np_default), encoding="utf-8")
    (WORK / "tokenizer_fertility.json").write_text(json.dumps(FERTILITY, indent=1))

load_aux(); save_aux()
removed = dedupe_runs(); log("duplicate run records removed:", removed)
consolidate_rankings()
write_readme()
ex = examples_text(); (WORK / "examples.txt").write_text(ex, encoding="utf-8"); print(ex[:6000])
summary = build_summary(); (WORK / "summary.json").write_text(summary, encoding="utf-8")
write_manifest()

zpath = WORK / "urdu_chunking_results.zip"
with zipfile.ZipFile(zpath, "w", zipfile.ZIP_DEFLATED, compresslevel=6) as z:
    for p in sorted(WORK.iterdir()):
        if p.is_file() and p.name != zpath.name: z.write(p, p.name)
log("zip:", zpath, f"{zpath.stat().st_size/1e6:.0f} MB")
print("\n=== SUMMARY_JSON_BEGIN ===\n" + summary + "\n=== SUMMARY_JSON_END ===")
print(f"\nSUMMARY size: {len(summary)/1024:.1f} KB | runs: {len(INDEX)}")
print("\nSELF-TESTS:"); [print(("PASS" if v["pass"] else "FAIL"), k) for k, v in TESTS.items()]
try:
    from google.colab import files
    # files.download(str(zpath))   # uncomment to download directly (large file); otherwise grab it from Drive
except Exception: pass

## DEVIATIONS (read this)

Static interpretation choices and deviations (runtime-detected ones are printed in the next cell and saved in `manifest.json`):

1. **Not executed by the author.** This notebook was written without running it. Use `DRY_RUN = True` first.
2. **Retrieval pool.** Retrieval searches the pooled chunks of the whole split (consistent with `documents=120, chunks=20219` in the schema). Chunks of other articles can be retrieved but are never relevant.
3. **Paragraphs in an article.** All paragraphs with a chosen Urdu variant are included (not only paragraphs holding retained questions). Counts are checked against the spec table.
4. **"Every 5th article is dev"** is ambiguous. The phase (`i % 5 == p`) whose dev question count is closest to 1,715 is used and reported.
5. **Budget metrics.** Primary `bm*` is strict (cumulative words never exceed the budget). `bm*x` (the last chunk may cross the budget) is stored as well, so the analysis can pick either definition. This is an addition, not a change of the primary metric.
6. **fixed_tokens in the BM25 grid.** The spec lists 5 strategies x sizes 50..800, so `size` is used as the token budget and the bge-m3 tokenizer is used for packing. Dense runs use the model's own tokenizer.
7. **Token counting for fixed_tokens** uses tokenizer offsets on the full document (exact up to boundary effects of 1-2 tokens), not a re-tokenization per candidate chunk. `budget_overflow_rate` reports how often a chunk really exceeds the budget.
8. **Char BM25**: words shorter than 3 characters are kept whole as one term (otherwise they would vanish). The grid auto-reduces to overlap 0 if projected over 2 h; this is recorded.
9. **Sentence/recursive overlap** is by whole units with total overlap <= `size*overlap` words, so small sizes often produce no overlap.
10. **English control** uses the same Urdu-terminator rules for `sentence_aware` and `recursive_urdu` (strict spec); on English text these mostly cut at paragraph breaks. Set `EN_BOUNDARIES_FOR_EN_CORPUS = True` for an English-aware variant. Only `recursive_english` is language-appropriate.
11. **Normalization runs** (N1/N2) keep arrays at full length; questions whose span cannot be mapped get `-1` and are listed in `excluded_question_positions`.
12. **Extra keys** in run records (`bm*x`, `chunk_words_mean`, `chunk_tokens_mean/p95`, `budget_overflow_rate`, `excluded_question_positions`) and extra files (`run_index.jsonl`, `examples.txt`, `tokenizer_fertility.json`, `questions_native.csv`).
13. **Splits.** Only `test` is run by default (the spec's 7,362-question summary). Dev is used for the prefix check, BM25 sanity and the audit sample.

In [ ]:
print("RUNTIME DEVIATIONS:" if DEVIATIONS else "No runtime deviations recorded.")
for d in DEVIATIONS: print(" -", d)